This notebook attempts to implement the order of purchases as a feature in our models. We will attempt similar models as prior notebooks, but using the order of purchases by each user and survey response IDs (which uniquely identify each user) as features.

In [ ]:
%pip install pandas matplotlib seaborn scikit-learn tensorflow

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

I0000 00:00:1776211858.292732    6539 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776211860.217426    6539 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776211863.939557    6539 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


We first need to add the survey response IDs and order date columns back into our data. Because we did not change the order of our data or expand any rows, we can simply add these features from the unencoded data to the encoded data, so we do not need to rerun all of our encoding steps.

In [116]:
data_unencoded = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')
data_encoded = pd.read_csv('/workspaces/group-project-bas-team/data/cleaned_data.csv')

In [117]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 156 entries, Purchase Price Per Unit to life-changes_Moved place of residence,Had a child
dtypes: float64(142), int64(14)
memory usage: 186.9 MB


In [118]:
data_unencoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                157026 non-null  str    
 1   Purchase Price Per Unit   157026 non-null  float64
 2   Quantity                  157026 non-null  int64  
 3   Shipping Address State    157026 non-null  str    
 4   Title                     157026 non-null  str    
 5   ASIN/ISBN (Product Code)  157026 non-null  str    
 6   Category                  157026 non-null  str    
 7   Survey ResponseID         157026 non-null  str    
 8   age                       157026 non-null  str    
 9   hispanic                  157026 non-null  str    
 10  race                      157026 non-null  str    
 11  education                 157026 non-null  str    
 12  income                    157026 non-null  str    
 13  gender                    157026 non-null  str    
 14 

In [119]:
data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]

/tmp/ipykernel_6539/601039707.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_encoded[['Survey ResponseID', 'Order Date']] = data_unencoded[['Survey ResponseID', 'Order Date']]


In [120]:
data_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 158 entries, Purchase Price Per Unit to Order Date
dtypes: float64(142), int64(14), str(2)
memory usage: 189.3 MB


In [121]:
# storing data_encoded as just data for easier code writing
data = data_encoded

In [122]:
# Ensuring the data is sorted by order date within each user's order history
data['Order Date'] = pd.to_datetime(data['Order Date'])
data = data.sort_values(by=['Survey ResponseID', 'Order Date'])

Next, we need to create a column that assigns a number to each order, representing the position in the sequence of a customer's orders.

In [123]:
data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)

/tmp/ipykernel_6539/693081789.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['Purchase_Order'] = data.groupby('Survey ResponseID')['Order Date'].rank(method='first').astype(int)


We will also add a column for time since last purchase

In [129]:
data['Days_Since_Last_Purchase'] = data.groupby('Survey ResponseID')['Order Date'].diff().dt.days.fillna(0)

In [130]:
data.head()

,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category,age,hispanic,education,income,howmany,...,"life-changes_Lost a job ,Moved place of residence,Became pregnant","life-changes_Lost a job ,Moved place of residence,Became pregnant,Had a child","life-changes_Lost a job ,Moved place of residence,Had a child",life-changes_Moved place of residence,"life-changes_Moved place of residence,Became pregnant,Had a child","life-changes_Moved place of residence,Had a child",Survey ResponseID,Order Date,Purchase_Order,Days_Since_Last_Purchase
0,7.98,1,83780,34099,586,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-04,1,0.0
1,13.99,1,16614,43470,729,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-22,2,18.0
2,10.45,1,73826,47343,432,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,3,3.0
3,10.00,1,77034,21023,1240,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2018-12-25,4,0.0
4,10.99,1,62681,39476,331,3,1,3,2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,R_01vNIayewjIIKMF,2019-02-18,5,55.0


In [131]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 160 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: datetime64[us](1), float64(143), int64(15), str(1)
memory usage: 191.7 MB


Next, we will remove the Order Date column, since we only needed it to order purchases by date and we already have columns for day/month/year.

In [132]:
data = data.drop(columns=['Order Date'])

Finally, we will encode the survey response ID column. Since there are thousands of unique IDs in our dataset, we will simply label encode this column.

The rationale for keeping this column is to distinguish multiple purchases that may have the same rank in purchase order (i.e., the first purchases made by different users will both be ranked 1). This is feasible because our training and testing sets are split by date, so each unique order ID will be present in both the training and testing sets.

In [133]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['Survey ResponseID'] = encoder.fit_transform(data['Survey ResponseID'])

data.info()

<class 'pandas.DataFrame'>
RangeIndex: 157026 entries, 0 to 157025
Columns: 159 entries, Purchase Price Per Unit to Days_Since_Last_Purchase
dtypes: float64(143), int64(16)
memory usage: 190.5 MB


Now we can attempt to build models using these new columns. We will start by replicating our MLP.

In [134]:
from sklearn.preprocessing import StandardScaler

train = data[data['order_year']<=2021]
test = data[data['order_year']>2021]

data_limited_train = train.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)
data_limited_test = test.drop(['Title', 'ASIN/ISBN (Product Code)'], axis=1)

X_train = data_limited_train.drop('Category', axis=1)
y_train = data_limited_train['Category']
X_train_scaled = StandardScaler().fit_transform(X_train)

X_test = data_limited_test.drop('Category', axis=1)
y_test = data_limited_test['Category']
X_test_scaled = StandardScaler().fit_transform(X_test)

In [135]:
X_train_scaled.shape[1]

156

In [136]:
y_train.max()

np.int64(1624)

In [187]:
import tensorflow as tf

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(156, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [188]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 300)            │        47,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241,325 (942.68 KB)

 Trainable params: 241,325 (942.68 KB)

 Non-trainable params: 0 (0.00 B)

In [189]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [190]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - accuracy: 0.0603 - loss: 6.5133 - val_accuracy: 0.0553 - val_loss: 6.2603 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0725 - loss: 6.0907 - val_accuracy: 0.0540 - val_loss: 6.1986 - learning_rate: 0.0100
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.0772 - loss: 6.0028 - val_accuracy: 0.0547 - val_loss: 6.1848 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0798 - loss: 5.9287 - val_accuracy: 0.0549 - val_loss: 6.1800 - learning_rate: 0.0100
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.0821 - loss: 5.8580 - val_accuracy: 0.0584 - val_loss: 6.1808 - learning_rate: 0.0100
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.0846 - loss: 5.7890 - val_accuracy: 0.0592 - val_loss: 6.1820 - learning_rate: 0.0100
Epoch 7/30
2833/2842 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.07

In [191]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 4s - 3ms/step - accuracy: 0.0527 - loss: 6.1303

Test accuracy: 0.05270802974700928


In [192]:
# Same model, using Adam optimizer instead of SGD
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(156, )))

model.add(tf.keras.layers.Dense(300, activation="relu")) 

model.add(tf.keras.layers.Dense(100, activation="relu"))

model.add(tf.keras.layers.Dense(1625, activation="softmax")) 

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [193]:
history = model.fit(X_train_scaled, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.0563 - loss: 6.2453 - val_accuracy: 0.0579 - val_loss: 6.4444 - learning_rate: 0.0010
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.0655 - loss: 5.9058 - val_accuracy: 0.0592 - val_loss: 6.4483 - learning_rate: 0.0010
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.0753 - loss: 5.6986 - val_accuracy: 0.0490 - val_loss: 6.5703 - learning_rate: 0.0010
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0661 - loss: 5.5498
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0795 - loss: 5.5353 - val_accuracy: 0.0585 - val_loss: 6.6454 - learning_rate: 0.0010
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.0886 - loss: 5.3918 - val_accuracy: 0.0546 - val_loss: 6.5601 - learning_rate: 5.0000e-04
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0

In [194]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 6s - 5ms/step - accuracy: 0.0428 - loss: 6.4171

Test accuracy: 0.04281662777066231


In [78]:
X_train_scaled.shape

(113655, 155)

In [196]:
X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], 1, 156))
X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], 1, 156))

In [195]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 156))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [197]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 150)            │       184,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       245,375 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 429,575 (1.64 MB)

 Trainable params: 429,575 (1.64 MB)

 Non-trainable params: 0 (0.00 B)

In [199]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

In [200]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.0498 - loss: 7.3100 - val_accuracy: 0.0612 - val_loss: 7.2191 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0540 - loss: 7.1422 - val_accuracy: 0.0612 - val_loss: 7.0633 - learning_rate: 0.0100
Epoch 3/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0540 - loss: 6.9936 - val_accuracy: 0.0612 - val_loss: 6.9480 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.0540 - loss: 6.8811 - val_accuracy: 0.0612 - val_loss: 6.8708 - learning_rate: 0.0100
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - accuracy: 0.0539 - loss: 6.7822 - val_accuracy: 0.0612 - val_loss: 6.8073 - learning_rate: 0.0100
Epoch 6/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.0597 - loss: 6.6834 - val_accuracy: 0.0611 - val_loss: 6.7521 - learning_rate: 0.0100
Epoch 7/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.0

In [201]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 4s - 3ms/step - accuracy: 0.0549 - loss: 6.1743

Test accuracy: 0.05489843338727951


In [204]:
# Adding more layers

tf.keras.backend.clear_session()
tf.random.set_seed(42)

model = tf.keras.Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(1, 156))) 

model.add(tf.keras.layers.LSTM(150, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=True)) 
model.add(tf.keras.layers.LSTM(100, return_sequences=False)) 

model.add(tf.keras.layers.Dense(1625, activation="softmax"))

In [205]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 150)         │       184,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 100)         │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        80,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1625)           │       164,125 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 529,125 (2.02 MB)

 Trainable params: 529,125 (2.02 MB)

 Non-trainable params: 0 (0.00 B)

In [206]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

In [207]:
history = model.fit(X_train_reshaped, y_train, epochs=30, shuffle=False, validation_split=0.2, callbacks=callbacks)

Epoch 1/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.0538 - loss: 7.3133 - val_accuracy: 0.0612 - val_loss: 7.2165 - learning_rate: 0.0100
Epoch 2/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 7.1603 - val_accuracy: 0.0612 - val_loss: 7.0557 - learning_rate: 0.0100
Epoch 3/30
2837/2842 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0457 - loss: 7.0868
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 7.0314 - val_accuracy: 0.0612 - val_loss: 6.9314 - learning_rate: 0.0100
Epoch 4/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 6.9583 - val_accuracy: 0.0612 - val_loss: 6.8846 - learning_rate: 0.0050
Epoch 5/30
2842/2842 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.0540 - loss: 6.9194 - val_accuracy: 0.0612 - val_loss: 6.8433 - learning_rate: 0.0050


In [208]:
test_loss, test_acc = model.evaluate(X_test_reshaped, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

1356/1356 - 5s - 4ms/step - accuracy: 0.0395 - loss: 7.2575

Test accuracy: 0.0395425520837307
